In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from utils import *
import prompt
from sklearn.cluster import KMeans
import openai
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# data

In [ ]:
config = load_config("configs/config_cluster_deconv.yaml")
name_truth = config.name_truth
print(config.replicate)
print(config.data_name)

In [ ]:
config.data_name = "151673"
config.refresh_paths()
print(config.data_name)

In [ ]:
# --- Load data ---
data_path = str(dataset_dir("visium_libd", config.data_name))
# marker_path = "data/reference/marker_genes/Mouse_cell_markers.txt"
adata = sc.read_visium(data_path)
adata.var_names_make_unique()

# Normalize data
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)

cell_proportion_data = pd.read_csv(os.path.join(data_path, f"celltype_proportions_{config.data_name}.csv"), index_col=0)

adata.obs = adata.obs.join(cell_proportion_data)

# read the metadata
meta_data = pd.read_csv(os.path.join(data_path, "metadata.tsv"), sep="\t")
# merge the metadata to adata
adata.obs = adata.obs.merge(meta_data, left_index=True, right_index=True, how="left")

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

# Rename the obs_names of adata
adata.obs_names = [f'spot_{i}' for i in range(len(adata.obs_names))]

# Transform spatial coordinates to DataFrame for sparse_adjacency
pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
cell_proportion_data = adata.obs[cell_proportion_data.columns].copy()

adj_matrix, distances = sparse_adjacency(pos_data, threshold=config.r, add_diagonal=True)


# Calculate the number of neighbors of each node
n_neighbors = adj_matrix.sum(axis=1).mean()
print(f"Number of neighbors: {sig_figs(n_neighbors, 3)}")

# If you want to store the n_neighbors of each node
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
adata.obs['n_neighbors'] = n_neighbors

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(cell_proportion_data)


# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

# transform the neighbor matrix to a dataframe
neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized, index=cell_proportion_data.index, 
                              columns=cell_proportion_data.columns)

# cluster 

In [ ]:
# use KMeans instead of KModes

km = KMeans(n_clusters=len(adata.obs[name_truth].unique()), random_state=42)
clusters = km.fit_predict(neighbor_normalized_df)
cluster_centers = km.cluster_centers_
cluster_centers = pd.DataFrame(cluster_centers, columns=neighbor_normalized_df.columns)
adata.obs['kmeans'] = clusters.astype(str)

In [ ]:
cluster_centers

In [ ]:
# plot initial clusters
sc.pl.spatial(adata, color=["kmeans", name_truth], size=1.4, show=False)
print(adjusted_rand_score(adata.obs["kmeans"], adata.obs[name_truth]))

# prompt
Important !!!!!!!!! name of niche

In [ ]:

domain_mapping = {1: "Layer1", 5: "Layer2", 6: "Layer3", 3: "Layer4", 0: "Layer5", 2: "Layer6", 4: "WM"}

cell_names_mapping = {'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}


config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

In [ ]:
# calculate prototype
one_shot_df = cluster_centers
print(one_shot_df.index)

# change the index of one_shot_df to be the same as the domain_mapping
one_shot_df.index = one_shot_df.index.map(config.domain_mapping)
print(one_shot_df.index)
# generate Comparison-based Prompt
config.oneshot_prompt = prompt.CP_celltype(one_shot_df, config)

In [ ]:
x = [i for i in range(len(adata)) if adata.obs[config.name_truth].iloc[i] == "Layer6"][30:31]
print(config.oneshot_prompt + prompt.oneshot_celltype(neighbor_normalized_df, x, config))


# GPT

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt.oneshot_celltype, batch_size = 5000, n_rows = 1)


In [ ]:
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_cluster_deconv.yaml > 151507_cluster_deconv.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['cluster_gpt4o_mini']

In [ ]:
gpt_results_df.loc['spot_3875'] = 'unknown'


In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.value_counts()

# Gemini

In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)

In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                neighbor_normalized_df, config, 
                                                prompt.oneshot_celltype, n_rows=1, 
                                                column_name="cluster_gemini")

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
gemini_results_df = pd.read_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)

In [ ]:
gemini_results_df.index.difference(adata.obs.index)

In [ ]:
gemini_results_df.loc['spot_1409'] = 'Layer5'

In [ ]:
gemini_results_df.value_counts()

# plot and save

In [ ]:
adata.obs = adata.obs.join(gemini_results_df)
adata.obs = adata.obs.join(gpt_results_df)
adata.obs['cluster_gemini'] = adata.obs['cluster_gemini'].fillna("unknown")
adata.obs['cluster_gpt4o_mini'] = adata.obs['cluster_gpt4o_mini'].fillna("unknown")
sc.pl.spatial(adata, color=["cluster_gemini", "cluster_gpt4o_mini", name_truth], size=1.4, show=False)
print(adjusted_rand_score(adata.obs["cluster_gemini"], adata.obs[name_truth]))
print(adjusted_rand_score(adata.obs["cluster_gpt4o_mini"], adata.obs[name_truth]))

In [ ]:
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")